# Projeto Desenvolvido na Data Science Academy

# Análise Exploratória de Dados e Modelagem
## Dataset: Telco Customer Churn

Este notebook realiza a análise exploratória dos dados e o treinamento do modelo de previsão de churn.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_preprocessing import dsa_load_data as load_data, dsa_handle_missing_values as handle_missing_values, dsa_encode_target as encode_target, dsa_prepare_features as prepare_features
from src.feature_engineering import dsa_build_preprocessor as build_preprocessor, dsa_get_feature_names as get_feature_names, dsa_identify_feature_types as identify_feature_types

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Carregamento dos Dados

In [ ]:
data_path = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'

df = load_data(data_path)

print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Análise de Missing Values

In [ ]:
# Verifica valores ausentes
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'ausentes': missing, 'percentual': missing_pct}).query('ausentes > 0')

In [ ]:
# Trata TotalCharges (espaços em branco)
if 'TotalCharges' in df.columns:
    empty_total = (df['TotalCharges'].astype(str).str.strip() == '').sum()
    print(f'TotalCharges com espaços vazios: {empty_total}')

## 3. Distribuição da Variável Alvo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Contagem
churn_counts = df['Churn'].value_counts()
axes[0].bar(churn_counts.index, churn_counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribuição de Churn')
axes[0].set_ylabel('Contagem')

for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Percentual
churn_pct = df['Churn'].value_counts(normalize=True) * 100
axes[1].pie(churn_pct.values, labels=churn_pct.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Percentual de Churn')

plt.tight_layout()
plt.show()

print(f'Taxa de churn: {churn_pct["Yes"]:.1f}%')

## 4. Distribuição das Features Numéricas

In [ ]:
df_clean = handle_missing_values(df)
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(numeric_cols):
    for label, color in [('No', '#2ecc71'), ('Yes', '#e74c3c')]:
        subset = df_clean[df_clean['Churn'] == label][col]
        axes[i].hist(subset, bins=30, alpha=0.6, label=f'Churn={label}', color=color)
    axes[i].set_title(f'Distribuição de {col}')
    axes[i].set_xlabel(col)
    axes[i].legend()

plt.tight_layout()
plt.show()

## 5. Análise de Features Categóricas

In [ ]:
cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'gender', 'Partner', 'Dependents']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df_clean[col], df_clean['Churn'], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[i], color=['#2ecc71', '#e74c3c'])
    axes[i].set_title(f'Churn por {col}')
    axes[i].set_ylabel('Percentual (%)')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].legend(title='Churn')

plt.tight_layout()
plt.show()

## 6. Correlações

In [ ]:
df_encoded = encode_target(df_clean.copy())
numeric_df = df_encoded[['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn']]

plt.figure(figsize=(8, 6))
sns.heatmap(numeric_df.corr(), annot=True, cmap='RdYlBu_r', center=0, fmt='.2f')
plt.title('Matriz de Correlação')
plt.tight_layout()
plt.show()

## 7. Treinamento do Modelo

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

from src.evaluate import dsa_evaluate_model as evaluate_model, dsa_get_feature_importance as get_feature_importance

# Prepara dados
df_model = encode_target(df_clean.copy())
X, y = prepare_features(df_model)

numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [c for c in X.columns if c not in numeric_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Treino: {X_train.shape[0]} | Teste: {X_test.shape[0]}')

In [ ]:
# Constroi e treina pipeline
preprocessor = build_preprocessor(numeric_features, categorical_features)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=5,
        min_samples_split=10, min_samples_leaf=5,
        subsample=0.8, random_state=42
    ))
])

pipeline.fit(X_train, y_train)
print('Treinamento concluído!')

## 8. Avaliação do Modelo

In [ ]:
# Métricas
metrics = evaluate_model(pipeline, X_test, y_test)
for metric, value in metrics.items():
    print(f'{metric:>12}: {value:.4f}')

In [ ]:
# Matriz de confusão
y_pred = pipeline.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Não Churn', 'Churn'],
            yticklabels=['Não Churn', 'Churn'])
plt.title('Matriz de Confusão')
plt.ylabel('Real')
plt.xlabel('Predito')
plt.tight_layout()
plt.show()

In [ ]:
# Curva ROC
y_proba = pipeline.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#3498db', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taxa de Falsos Positivos')
plt.ylabel('Taxa de Verdadeiros Positivos')
plt.title('Curva ROC')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
feature_names = get_feature_names(pipeline.named_steps['preprocessor'], X_train)
feat_imp = get_feature_importance(pipeline, feature_names)

top_10 = feat_imp[:10]
names = [f['feature'] for f in top_10]
values = [f['importance'] for f in top_10]

plt.figure(figsize=(10, 6))
plt.barh(range(len(names)), values, color='#3498db')
plt.yticks(range(len(names)), names)
plt.xlabel('Importância')
plt.title('Top 10 Features Mais Importantes')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Conclusão

O modelo Gradient Boosting Classifier foi treinado e avaliado com sucesso.
As features mais importantes para previsão de churn são tipicamente:
- **tenure** (tempo de permanência)
- **MonthlyCharges** (cobranças mensais)
- **Contract** (tipo de contrato)
- **TotalCharges** (cobranças totais)

O modelo serializado pode ser utilizado pela API para predições em tempo real.